# 07 — Profile POSet Main

This notebook comes after:

`06_Pre_POSet_EDA_Checks.ipynb`

Purpose:

Run the main **profile-level POSet analysis** using direction-aligned candidate variables.

This notebook does:

- read the Pre-POSet direction-aligned snapshot;
- read candidate variable sets;
- run profile-level POSet separately for 2007 and 2019 shocks;
- discretize continuous variables into ordinal profiles;
- construct profile dominance relations;
- compute Hasse edges through transitive reduction;
- identify Pareto profiles and Pareto countries;
- compute layers, comparability, and incomparability;
- export country-profile maps, profile tables, dominance relations, Hasse-ready edges, and diagrams.

This notebook does **not**:

- run epsilon sensitivity;
- create a final scalar Economic Resilience Index;
- use recovery as an ordering variable;
- claim causality between structure and recovery.

Important:

Recovery is kept only as an attached validation/outcome variable for later interpretation.  
It is not used to build the POSet relation.

## v2 correction

This version fixes the issue found after the first run:

- Some output tables already contain metadata columns like `shock_id`.
- The v1 runner tried to insert those metadata columns again, causing every POSet run to fail.
- v2 now adds metadata only when missing, and overwrites existing metadata values safely.


In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"

INPUT_SNAPSHOT_FILE = PRE_POSET_DIR / "pre_poset_analysis_ready_snapshot_full.csv"
CANDIDATE_VARIABLE_SETS_FILE = PRE_POSET_DIR / "candidate_variable_sets.csv"
CANDIDATE_SCORECARD_FILE = PRE_POSET_DIR / "candidate_variable_scorecard.csv"
BASELINE_COMPLETE_CASE_FILE = PRE_POSET_DIR / "baseline_complete_case_sample_by_shock.csv"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "07_Profile_POSet_Main"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Profile_POSet_Main"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Input snapshot file:", INPUT_SNAPSHOT_FILE.resolve())
print("Candidate variable sets file:", CANDIDATE_VARIABLE_SETS_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Figures folder:", FIGURES_DIR.resolve())

Run ID: 20260622_070814
Input snapshot file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\pre_poset_analysis_ready_snapshot_full.csv
Candidate variable sets file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\candidate_variable_sets.csv
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Profile_POSet_Main
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\07_Profile_POSet_Main
Figures folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Profile_POSet_Main


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

# Working default:
# We run the baseline 6-variable set first using complete cases.
# This is not a final methodological lock-in; it is a clean baseline so the code runs end-to-end.
MAIN_SET_NAME = "baseline_6_variables"

# Run these discretization levels for the main baseline set.
# Level 5 is treated as the working main output, but 3 and 4 are exported too.
MAIN_PROFILE_LEVELS_TO_RUN = [3, 4, 5]
WORKING_MAIN_PROFILE_LEVEL = 5

# Sensitivity-like profile runs inside this notebook.
# These are not the final sensitivity analysis notebook, but they help check whether the code works.
RUN_NON_BASELINE_SETS_AT_LEVELS = [5]

SHOCK_IDS = ["2007", "2019"]

# Sample strategy:
# complete_case means each POSet run uses countries with all variables in that run and, by default, Recovery available.
SAMPLE_STRATEGY = "complete_case"

# Recovery is validation-only, not an ordering variable.
# Keeping it available in the run sample makes downstream validation easier.
REQUIRE_RECOVERY_AVAILABLE_FOR_PROFILE_RUNS = True

# Discretization strategy:
# rank_quantile creates ordinal levels based on within-run ranks.
# Higher values receive higher levels.
DISCRETIZATION_METHOD = "rank_quantile"

# Skip diagrams for very large profile counts to avoid unreadable images.
MAX_PROFILE_NODES_FOR_DIAGRAM = 60

# Default fallback variable set if candidate_variable_sets.csv is unavailable.
FALLBACK_BASELINE_VARIABLES = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
    "governance_capacity_score_0_100",
]

print("Main set:", MAIN_SET_NAME)
print("Main profile levels:", MAIN_PROFILE_LEVELS_TO_RUN)
print("Working main profile level:", WORKING_MAIN_PROFILE_LEVEL)
print("Sample strategy:", SAMPLE_STRATEGY)
print("Require recovery available:", REQUIRE_RECOVERY_AVAILABLE_FOR_PROFILE_RUNS)

Main set: baseline_6_variables
Main profile levels: [3, 4, 5]
Working main profile level: 5
Sample strategy: complete_case
Require recovery available: True


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def clean_country_shock(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()

    return out


def parse_variable_list(value):
    if pd.isna(value):
        return []

    return [
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    ]


def variable_applicable_to_shock(variable, shock_id):
    variable = str(variable)
    shock_id = str(shock_id)

    match = re.search(r"_pre_(\d{4})", variable)

    if match:
        return match.group(1) == shock_id

    return True


def assign_rank_quantile_levels(series, n_levels):
    values = pd.to_numeric(series, errors="coerce")
    result = pd.Series(np.nan, index=series.index)

    non_missing = values.dropna()

    if non_missing.empty:
        return result

    if non_missing.nunique() == 1:
        midpoint = int(np.ceil(n_levels / 2))
        result.loc[non_missing.index] = midpoint
        return result.astype("Int64")

    pct_rank = non_missing.rank(method="average", pct=True)
    levels = np.ceil(pct_rank * n_levels).astype(int).clip(1, n_levels)

    result.loc[non_missing.index] = levels

    return result.astype("Int64")


def build_profile_code(row, level_columns):
    values = []

    for col in level_columns:
        value = row[col]

        if pd.isna(value):
            values.append("NA")
        else:
            values.append(str(int(value)))

    return "-".join(values)


def strict_profile_dominance(matrix):
    # matrix: rows are profiles, columns are ordinal dimensions.
    n = matrix.shape[0]
    dominance = np.zeros((n, n), dtype=bool)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            ge_all = np.all(matrix[i, :] >= matrix[j, :])
            gt_any = np.any(matrix[i, :] > matrix[j, :])

            if ge_all and gt_any:
                dominance[i, j] = True

    return dominance


def transitive_reduction_from_dominance(dominance):
    n = dominance.shape[0]
    hasse_edges = []

    for i in range(n):
        for j in range(n):
            if not dominance[i, j]:
                continue

            transitive = False

            for k in range(n):
                if k == i or k == j:
                    continue

                if dominance[i, k] and dominance[k, j]:
                    transitive = True
                    break

            if not transitive:
                hasse_edges.append((i, j))

    return hasse_edges


def assign_poset_layers(dominance):
    n = dominance.shape[0]
    remaining = set(range(n))
    layers = {i: np.nan for i in range(n)}
    layer = 1

    while remaining:
        nondominated = []

        for i in remaining:
            dominated_by_remaining = any(
                dominance[j, i]
                for j in remaining
                if j != i
            )

            if not dominated_by_remaining:
                nondominated.append(i)

        if not nondominated:
            raise ValueError("No nondominated nodes found. The relation may contain a cycle.")

        for i in nondominated:
            layers[i] = layer

        remaining -= set(nondominated)
        layer += 1

    return layers


def profile_comparability_metrics(dominance):
    n = dominance.shape[0]
    total_pairs = n * (n - 1) / 2

    if total_pairs == 0:
        return {
            "profile_count": n,
            "comparable_pairs": 0,
            "incomparable_pairs": 0,
            "comparability_ratio": np.nan,
            "incomparability_ratio": np.nan,
        }

    comparable_pairs = 0

    for i in range(n):
        for j in range(i + 1, n):
            if dominance[i, j] or dominance[j, i]:
                comparable_pairs += 1

    incomparable_pairs = int(total_pairs - comparable_pairs)

    return {
        "profile_count": n,
        "comparable_pairs": comparable_pairs,
        "incomparable_pairs": incomparable_pairs,
        "comparability_ratio": comparable_pairs / total_pairs,
        "incomparability_ratio": incomparable_pairs / total_pairs,
    }


def make_hasse_diagram(profile_table, hasse_edges, run_id_slug, title):
    if len(profile_table) == 0:
        return None

    if len(profile_table) > MAX_PROFILE_NODES_FOR_DIAGRAM:
        return None

    fig, ax = plt.subplots(figsize=(14, max(8, profile_table["poset_layer"].max() * 1.6)))

    profile_table = profile_table.copy()
    positions = {}

    for layer, group in profile_table.groupby("poset_layer"):
        group = group.sort_values(["profile_code", "profile_id"]).reset_index(drop=True)
        count = len(group)

        if count == 1:
            xs = [0.0]
        else:
            xs = np.linspace(-count / 2, count / 2, count)

        y = -float(layer)

        for x, (_, row) in zip(xs, group.iterrows()):
            positions[row["profile_id"]] = (float(x), y)

    for source_id, target_id in hasse_edges:
        if source_id not in positions or target_id not in positions:
            continue

        x1, y1 = positions[source_id]
        x2, y2 = positions[target_id]

        ax.annotate(
            "",
            xy=(x2, y2 + 0.08),
            xytext=(x1, y1 - 0.08),
            arrowprops=dict(arrowstyle="->", lw=0.8, alpha=0.65),
        )

    for _, row in profile_table.iterrows():
        x, y = positions[row["profile_id"]]
        ax.scatter(x, y, s=850, zorder=3)

        label = (
            f"{row['profile_id']}\n"
            f"{row['profile_code']}\n"
            f"n={int(row['country_count'])}"
        )

        ax.text(
            x,
            y,
            label,
            ha="center",
            va="center",
            fontsize=8,
            zorder=4,
        )

    ax.set_title(title, fontsize=14, pad=18)
    ax.set_xlabel("Profiles within layer")
    ax.set_ylabel("POSet layer: 1 = Pareto / nondominated")
    ax.grid(axis="y", alpha=0.2)
    yticks = sorted([-float(layer) for layer in profile_table["poset_layer"].unique()])
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(int(abs(y))) for y in yticks])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    output_path = FIGURES_DIR / f"{run_id_slug}_hasse_diagram.png"
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return output_path


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))

In [4]:
# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

require_file(INPUT_SNAPSHOT_FILE, "Pre-POSet analysis-ready snapshot")

snapshot = pd.read_csv(INPUT_SNAPSHOT_FILE)
snapshot = clean_country_shock(snapshot)

if "country_name" not in snapshot.columns:
    snapshot["country_name"] = snapshot["Country"]

required_cols = {"Country", "shock_id", "Recovery"}
missing_required = required_cols - set(snapshot.columns)

if missing_required:
    raise ValueError(f"Input snapshot missing required columns: {missing_required}")

if CANDIDATE_VARIABLE_SETS_FILE.exists():
    candidate_variable_sets = pd.read_csv(CANDIDATE_VARIABLE_SETS_FILE)
else:
    candidate_variable_sets = pd.DataFrame([{
        "set_name": MAIN_SET_NAME,
        "intended_use": "main_candidate",
        "shock_specific": False,
        "variables": ",".join(FALLBACK_BASELINE_VARIABLES),
        "variable_count": len(FALLBACK_BASELINE_VARIABLES),
        "note": "Fallback baseline set because candidate_variable_sets.csv was not found.",
    }])

if CANDIDATE_SCORECARD_FILE.exists():
    candidate_scorecard = pd.read_csv(CANDIDATE_SCORECARD_FILE)
else:
    candidate_scorecard = pd.DataFrame()

if BASELINE_COMPLETE_CASE_FILE.exists():
    baseline_complete_case_sample = pd.read_csv(BASELINE_COMPLETE_CASE_FILE)
    baseline_complete_case_sample = clean_country_shock(baseline_complete_case_sample)
else:
    baseline_complete_case_sample = pd.DataFrame()

input_summary = pd.DataFrame([{
    "rows": len(snapshot),
    "countries": snapshot["Country"].nunique(),
    "shock_ids": ",".join(sorted(snapshot["shock_id"].unique())),
    "candidate_variable_sets": len(candidate_variable_sets),
    "baseline_complete_case_rows": len(baseline_complete_case_sample),
    "run_timestamp": RUN_TIMESTAMP,
}])

input_summary.to_csv(
    DIAGNOSTICS_DIR / "profile_poset_input_summary.csv",
    index=False,
)

print("Snapshot rows:", len(snapshot))
print("Countries:", snapshot["Country"].nunique())
print("Shock IDs:", sorted(snapshot["shock_id"].unique()))
print("Candidate variable sets:")
display(candidate_variable_sets)
print("Candidate scorecard:")
display(candidate_scorecard.head(50))

Snapshot rows: 300
Countries: 162
Shock IDs: ['2007', '2019']
Candidate variable sets:


,set_name,intended_use,shock_specific,variables,variable_count,note
0,baseline_6_variables,main_candidate,False,"debt_capacity_score_0_100,employment_strength_...",6,"Balanced structural set across fiscal, labour,..."
1,baseline_plus_gdp_stability_2007,shock_2007_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2007 GDP growth stability. Use only f...
2,baseline_plus_gdp_stability_2019,shock_2019_sensitivity,True,"debt_capacity_score_0_100,employment_strength_...",7,Adds pre-2019 GDP growth stability. Use only f...
3,sensitivity_remove_governance,sensitivity,False,"debt_capacity_score_0_100,employment_strength_...",5,Tests dependence on derived governance composite.
4,sensitivity_replace_debt_with_productivity,sensitivity,False,"employment_strength_score_0_100,rd_intensity_s...",6,Tests whether productive capacity changes orde...


Candidate scorecard:


,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,direction,reason,recommended_use,min_coverage_rate_broad_master,mean_coverage_rate_broad_master,min_countries_non_missing_broad_master,min_analysis_coverage_rate,mean_analysis_coverage_rate,min_analysis_countries_non_missing,shocks_available,max_abs_correlation,max_redundancy_class,pre_poset_recommendation,warning
0,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,higher_is_better,Lower energy import dependency means higher en...,main_or_baseline,0.8896,0.8900,130,0.9767,0.9770,42,2,0.3162,lower_redundancy_risk,recommended_baseline_pool,NaN
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,higher_is_better,Adult tertiary attainment captures accumulated...,main_or_baseline,0.2534,0.2728,37,0.8409,0.8972,37,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,NaN
2,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,higher_is_better,R&D/GDP captures innovation capacity and is al...,main_or_baseline,0.2922,0.2968,44,0.8636,0.8737,38,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,NaN
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,higher_is_better,Derived WGI governance composite summarizes in...,main_or_baseline,0.9863,0.9932,144,1.0000,1.0000,43,2,0.5497,lower_redundancy_risk,recommended_baseline_pool,NaN
4,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,higher_is_better,Lower unemployment indicates stronger labour-m...,main_or_baseline,0.3356,0.3366,49,0.9545,0.9656,42,2,0.4342,lower_redundancy_risk,recommended_baseline_pool,NaN
5,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.3014,0.3014,44,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,recommended_baseline_pool,NaN
6,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.2857,0.2857,44,1.0000,1.0000,43,1,0.7370,moderate_redundancy_risk,recommended_baseline_pool,NaN
7,unemployment_stability_pre_2007_score_0_100,unemployment_stability_pre_2007_aligned_raw,unemployment_stability_negative_std_pre_2007,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3288,0.3288,48,0.9545,0.9545,42,1,0.4128,lower_redundancy_risk,review_candidate,NaN
8,unemployment_stability_pre_2019_score_0_100,unemployment_stability_pre_2019_aligned_raw,unemployment_stability_negative_std_pre_2019,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3312,0.3312,51,0.9535,0.9535,41,1,0.6411,lower_redundancy_risk,review_candidate,NaN
9,inflation_stability_pre_2007_score_0_100,inflation_stability_pre_2007_aligned_raw,inflation_stability_negative_std_pre_2007,Inflation stability,macro_stability,review_candidate,higher_is_better,Pre-shock inflation volatility converted to st...,review_or_sensitivity,0.3219,0.3219,47,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,review_candidate,NaN


In [5]:
# ------------------------------------------------------
# BUILD PROFILE RUN PLAN
# ------------------------------------------------------

run_plan_rows = []

for _, row in candidate_variable_sets.iterrows():
    set_name = str(row["set_name"])
    variables = parse_variable_list(row["variables"])
    intended_use = row.get("intended_use", "")
    shock_specific = bool(row.get("shock_specific", False))
    note = row.get("note", "")

    if not variables:
        continue

    # Decide which shocks this set applies to.
    if set_name.endswith("_2007"):
        shocks_for_set = ["2007"]
    elif set_name.endswith("_2019"):
        shocks_for_set = ["2019"]
    else:
        shocks_for_set = SHOCK_IDS

    if set_name == MAIN_SET_NAME:
        profile_levels_to_run = MAIN_PROFILE_LEVELS_TO_RUN
    else:
        profile_levels_to_run = RUN_NON_BASELINE_SETS_AT_LEVELS

    for shock_id in shocks_for_set:
        applicable_variables = [
            var for var in variables
            if variable_applicable_to_shock(var, shock_id)
        ]

        missing_variables = [
            var for var in applicable_variables
            if var not in snapshot.columns
        ]

        for n_levels in profile_levels_to_run:
            run_plan_rows.append({
                "run_key": f"{set_name}__shock_{shock_id}__levels_{n_levels}",
                "set_name": set_name,
                "intended_use": intended_use,
                "shock_id": shock_id,
                "profile_levels": n_levels,
                "variables": ",".join(applicable_variables),
                "variable_count": len(applicable_variables),
                "missing_variables": ",".join(missing_variables),
                "missing_variable_count": len(missing_variables),
                "sample_strategy": SAMPLE_STRATEGY,
                "require_recovery_available": REQUIRE_RECOVERY_AVAILABLE_FOR_PROFILE_RUNS,
                "is_working_main_run": (
                    set_name == MAIN_SET_NAME
                    and n_levels == WORKING_MAIN_PROFILE_LEVEL
                ),
                "note": note,
            })

profile_run_plan = pd.DataFrame(run_plan_rows)

profile_run_plan.to_csv(
    PROCESSED_DIR / "profile_run_plan.csv",
    index=False,
)

profile_run_plan.to_csv(
    DIAGNOSTICS_DIR / "profile_run_plan.csv",
    index=False,
)

display(profile_run_plan)

,run_key,set_name,intended_use,shock_id,profile_levels,variables,variable_count,missing_variables,missing_variable_count,sample_strategy,require_recovery_available,is_working_main_run,note
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,main_candidate,2007,3,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,False,"Balanced structural set across fiscal, labour,..."
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,main_candidate,2007,4,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,False,"Balanced structural set across fiscal, labour,..."
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,main_candidate,2007,5,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,main_candidate,2019,3,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,False,"Balanced structural set across fiscal, labour,..."
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,main_candidate,2019,4,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,False,"Balanced structural set across fiscal, labour,..."
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,main_candidate,2019,5,"debt_capacity_score_0_100,employment_strength_...",6,,0,complete_case,True,True,"Balanced structural set across fiscal, labour,..."
6,baseline_plus_gdp_stability_2007__shock_2007__...,baseline_plus_gdp_stability_2007,shock_2007_sensitivity,2007,5,"debt_capacity_score_0_100,employment_strength_...",7,,0,complete_case,True,False,Adds pre-2007 GDP growth stability. Use only f...
7,baseline_plus_gdp_stability_2019__shock_2019__...,baseline_plus_gdp_stability_2019,shock_2019_sensitivity,2019,5,"debt_capacity_score_0_100,employment_strength_...",7,,0,complete_case,True,False,Adds pre-2019 GDP growth stability. Use only f...
8,sensitivity_remove_governance__shock_2007__lev...,sensitivity_remove_governance,sensitivity,2007,5,"debt_capacity_score_0_100,employment_strength_...",5,,0,complete_case,True,False,Tests dependence on derived governance composite.
9,sensitivity_remove_governance__shock_2019__lev...,sensitivity_remove_governance,sensitivity,2019,5,"debt_capacity_score_0_100,employment_strength_...",5,,0,complete_case,True,False,Tests dependence on derived governance composite.


In [6]:
# ------------------------------------------------------
# MAIN PROFILE POSET RUNNER
# ------------------------------------------------------

def run_profile_poset(run_config, snapshot):
    run_key = run_config["run_key"]
    set_name = run_config["set_name"]
    shock_id = str(run_config["shock_id"])
    n_levels = int(run_config["profile_levels"])
    variables = parse_variable_list(run_config["variables"])

    missing_variables = [var for var in variables if var not in snapshot.columns]

    if missing_variables:
        raise ValueError(f"Run {run_key} has missing variables: {missing_variables}")

    run_df = snapshot[snapshot["shock_id"].astype(str) == shock_id].copy()

    if REQUIRE_RECOVERY_AVAILABLE_FOR_PROFILE_RUNS and "Recovery" in run_df.columns:
        run_df = run_df[run_df["Recovery"].notna()].copy()

    # Complete-case on ordering variables.
    run_df = run_df.dropna(subset=variables).copy()

    if run_df.empty:
        raise ValueError(f"Run {run_key} has zero rows after filtering.")

    # Discretize variables.
    level_columns = []

    for var in variables:
        level_col = f"{var}__L{n_levels}"
        run_df[level_col] = assign_rank_quantile_levels(run_df[var], n_levels)
        level_columns.append(level_col)

    run_df = run_df.dropna(subset=level_columns).copy()

    for col in level_columns:
        run_df[col] = run_df[col].astype(int)

    run_df["profile_code"] = run_df.apply(
        lambda row: build_profile_code(row, level_columns),
        axis=1,
    )

    run_df["profile_dimension_count"] = len(level_columns)

    # Profile table.
    country_summary = (
        run_df
        .groupby("profile_code")
        .agg(
            country_count=("Country", "nunique"),
            countries=("Country", lambda x: ";".join(sorted(x.unique()))),
            country_names=("country_name", lambda x: ";".join(sorted(set(map(str, x))))),
            recovery_median=("Recovery", "median"),
            recovery_mean=("Recovery", "mean"),
            recovery_min=("Recovery", "min"),
            recovery_max=("Recovery", "max"),
        )
        .reset_index()
    )

    profile_dimensions = (
        run_df
        .groupby("profile_code")[level_columns]
        .first()
        .reset_index()
    )

    profile_table = profile_dimensions.merge(
        country_summary,
        on="profile_code",
        how="left",
    )

    # Sort strongest-looking profiles first for stable IDs.
    profile_table["profile_digit_sum"] = profile_table[level_columns].sum(axis=1)
    profile_table = profile_table.sort_values(
        ["profile_digit_sum"] + level_columns,
        ascending=[False] + [False] * len(level_columns),
    ).reset_index(drop=True)

    profile_table["profile_id"] = [
        f"P{i + 1:03d}" for i in range(len(profile_table))
    ]

    profile_table = profile_table[
        ["profile_id", "profile_code"]
        + level_columns
        + [
            "profile_digit_sum",
            "country_count",
            "countries",
            "country_names",
            "recovery_median",
            "recovery_mean",
            "recovery_min",
            "recovery_max",
        ]
    ]

    # Dominance matrix.
    matrix = profile_table[level_columns].to_numpy(dtype=int)
    dominance = strict_profile_dominance(matrix)

    dominates_count = dominance.sum(axis=1)
    dominated_by_count = dominance.sum(axis=0)

    profile_table["dominates_profile_count"] = dominates_count
    profile_table["dominated_by_profile_count"] = dominated_by_count
    profile_table["is_pareto_profile"] = profile_table["dominated_by_profile_count"] == 0

    # Layers.
    layer_map = assign_poset_layers(dominance)
    profile_table["poset_layer"] = profile_table.index.map(layer_map).astype(int)

    # Hasse edges.
    edge_indices = transitive_reduction_from_dominance(dominance)

    hasse_edges = pd.DataFrame([
        {
            "source_profile_index": i,
            "target_profile_index": j,
            "source_profile_id": profile_table.loc[i, "profile_id"],
            "target_profile_id": profile_table.loc[j, "profile_id"],
            "source_profile_code": profile_table.loc[i, "profile_code"],
            "target_profile_code": profile_table.loc[j, "profile_code"],
            "source_layer": profile_table.loc[i, "poset_layer"],
            "target_layer": profile_table.loc[j, "poset_layer"],
        }
        for i, j in edge_indices
    ])

    if hasse_edges.empty:
        hasse_edges = pd.DataFrame(columns=[
            "source_profile_index",
            "target_profile_index",
            "source_profile_id",
            "target_profile_id",
            "source_profile_code",
            "target_profile_code",
            "source_layer",
            "target_layer",
        ])

    # Full dominance relation.
    dominance_rows = []

    for i in range(dominance.shape[0]):
        for j in range(dominance.shape[1]):
            if dominance[i, j]:
                dominance_rows.append({
                    "source_profile_id": profile_table.loc[i, "profile_id"],
                    "target_profile_id": profile_table.loc[j, "profile_id"],
                    "source_profile_code": profile_table.loc[i, "profile_code"],
                    "target_profile_code": profile_table.loc[j, "profile_code"],
                    "source_layer": profile_table.loc[i, "poset_layer"],
                    "target_layer": profile_table.loc[j, "poset_layer"],
                })

    dominance_relations = pd.DataFrame(dominance_rows)

    if dominance_relations.empty:
        dominance_relations = pd.DataFrame(columns=[
            "source_profile_id",
            "target_profile_id",
            "source_profile_code",
            "target_profile_code",
            "source_layer",
            "target_layer",
        ])

    # Incomparability pairs.
    incomparability_rows = []

    for i in range(dominance.shape[0]):
        for j in range(i + 1, dominance.shape[1]):
            if not dominance[i, j] and not dominance[j, i]:
                incomparability_rows.append({
                    "profile_a_id": profile_table.loc[i, "profile_id"],
                    "profile_b_id": profile_table.loc[j, "profile_id"],
                    "profile_a_code": profile_table.loc[i, "profile_code"],
                    "profile_b_code": profile_table.loc[j, "profile_code"],
                    "profile_a_layer": profile_table.loc[i, "poset_layer"],
                    "profile_b_layer": profile_table.loc[j, "poset_layer"],
                    "profile_a_countries": profile_table.loc[i, "countries"],
                    "profile_b_countries": profile_table.loc[j, "countries"],
                })

    incomparability_pairs = pd.DataFrame(incomparability_rows)

    if incomparability_pairs.empty:
        incomparability_pairs = pd.DataFrame(columns=[
            "profile_a_id",
            "profile_b_id",
            "profile_a_code",
            "profile_b_code",
            "profile_a_layer",
            "profile_b_layer",
            "profile_a_countries",
            "profile_b_countries",
        ])

    # Country-profile map.
    country_profile_map = run_df[
        ["Country", "country_name", "shock_id", "analysis_year", "Recovery", "profile_code"]
        + variables
        + level_columns
    ].copy()

    country_profile_map = country_profile_map.merge(
        profile_table[
            [
                "profile_id",
                "profile_code",
                "profile_digit_sum",
                "country_count",
                "dominates_profile_count",
                "dominated_by_profile_count",
                "is_pareto_profile",
                "poset_layer",
            ]
        ],
        on="profile_code",
        how="left",
    )

    pareto_profiles = profile_table[profile_table["is_pareto_profile"]].copy()

    pareto_countries = country_profile_map[
        country_profile_map["is_pareto_profile"]
    ].copy()

    layer_summary = (
        profile_table
        .groupby("poset_layer")
        .agg(
            profile_count=("profile_id", "nunique"),
            countries_in_layer=("country_count", "sum"),
            profiles=("profile_id", lambda x: ";".join(x)),
        )
        .reset_index()
        .sort_values("poset_layer")
    )

    metrics = profile_comparability_metrics(dominance)

    run_summary = {
        "run_key": run_key,
        "set_name": set_name,
        "shock_id": shock_id,
        "profile_levels": n_levels,
        "sample_strategy": SAMPLE_STRATEGY,
        "require_recovery_available": REQUIRE_RECOVERY_AVAILABLE_FOR_PROFILE_RUNS,
        "country_count": run_df["Country"].nunique(),
        "profile_count": len(profile_table),
        "variable_count": len(variables),
        "variables": ",".join(variables),
        "pareto_profile_count": int(profile_table["is_pareto_profile"].sum()),
        "pareto_country_count": int(pareto_countries["Country"].nunique()),
        "dominance_relation_count": len(dominance_relations),
        "hasse_edge_count": len(hasse_edges),
        "layer_count": int(profile_table["poset_layer"].max()),
        "max_country_count_per_profile": int(profile_table["country_count"].max()),
        "mean_country_count_per_profile": float(profile_table["country_count"].mean()),
        **metrics,
        "is_working_main_run": bool(run_config["is_working_main_run"]),
    }

    # Add run metadata to every table.
    metadata_cols = {
        "run_key": run_key,
        "set_name": set_name,
        "shock_id": shock_id,
        "profile_levels": n_levels,
    }

    tables = {
        "country_profile_map": country_profile_map,
        "profile_table": profile_table,
        "dominance_relations": dominance_relations,
        "hasse_edges": hasse_edges,
        "pareto_profiles": pareto_profiles,
        "pareto_countries": pareto_countries,
        "layer_summary": layer_summary,
        "incomparability_pairs": incomparability_pairs,
    }

    for name, table in tables.items():
        # v2 fix:
        # Some tables, especially country_profile_map, already contain shock_id.
        # Do not insert duplicate columns. Set the value if present; insert only if missing.
        for col, val in metadata_cols.items():
            if col in table.columns:
                table[col] = val
            else:
                table.insert(0, col, val)

        # Put metadata columns first in a stable order.
        metadata_order = ["run_key", "set_name", "shock_id", "profile_levels"]
        ordered_cols = [col for col in metadata_order if col in table.columns]
        ordered_cols += [col for col in table.columns if col not in ordered_cols]
        tables[name] = table[ordered_cols].copy()

    run_id_slug = re.sub(r"[^A-Za-z0-9_]+", "_", run_key)

    diagram_path = make_hasse_diagram(
        profile_table=profile_table,
        hasse_edges=[
            (int(row["source_profile_index"]), int(row["target_profile_index"]))
            for _, row in hasse_edges.iterrows()
        ],
        run_id_slug=run_id_slug,
        title=f"Profile POSet Hasse Diagram — {set_name}, shock {shock_id}, {n_levels} levels",
    )

    run_summary["hasse_diagram_path"] = str(diagram_path) if diagram_path is not None else ""

    return run_summary, tables

In [7]:
# ------------------------------------------------------
# EXECUTE PROFILE POSET RUNS
# ------------------------------------------------------

run_summaries = []
combined_tables = {
    "country_profile_map": [],
    "profile_table": [],
    "dominance_relations": [],
    "hasse_edges": [],
    "pareto_profiles": [],
    "pareto_countries": [],
    "layer_summary": [],
    "incomparability_pairs": [],
}

run_errors = []

for _, run_config in profile_run_plan.iterrows():
    run_key = run_config["run_key"]

    if run_config["missing_variable_count"] > 0:
        run_errors.append({
            "run_key": run_key,
            "error": f"Missing variables: {run_config['missing_variables']}",
        })
        continue

    try:
        run_summary, tables = run_profile_poset(run_config, snapshot)
        run_summaries.append(run_summary)

        for table_name, table in tables.items():
            combined_tables[table_name].append(table)

        print(f"Completed: {run_key}")

    except Exception as e:
        run_errors.append({
            "run_key": run_key,
            "error": str(e),
        })
        print(f"FAILED: {run_key} -> {e}")

profile_run_summary = pd.DataFrame(run_summaries)

expected_summary_cols = [
    "run_key", "set_name", "shock_id", "profile_levels", "sample_strategy",
    "require_recovery_available", "country_count", "profile_count",
    "variable_count", "variables", "pareto_profile_count", "pareto_country_count",
    "dominance_relation_count", "hasse_edge_count", "layer_count",
    "max_country_count_per_profile", "mean_country_count_per_profile",
    "comparable_pairs", "incomparable_pairs", "comparability_ratio",
    "incomparability_ratio", "is_working_main_run", "hasse_diagram_path",
]

if profile_run_summary.empty:
    profile_run_summary = pd.DataFrame(columns=expected_summary_cols)

profile_run_errors = pd.DataFrame(run_errors)

if profile_run_errors.empty:
    profile_run_errors = pd.DataFrame(columns=["run_key", "error"])

profile_run_summary.to_csv(
    PROCESSED_DIR / "profile_run_summary.csv",
    index=False,
)

profile_run_summary.to_csv(
    DIAGNOSTICS_DIR / "profile_run_summary.csv",
    index=False,
)

profile_run_errors.to_csv(
    DIAGNOSTICS_DIR / "profile_run_errors.csv",
    index=False,
)

if not profile_run_errors.empty:
    display(profile_run_errors)

if profile_run_summary.empty:
    print("WARNING: No profile POSet runs completed. Check profile_run_errors.csv.")
else:
    print("Completed runs:", len(profile_run_summary))

display(profile_run_summary)

Completed: baseline_6_variables__shock_2007__levels_3
Completed: baseline_6_variables__shock_2007__levels_4
Completed: baseline_6_variables__shock_2007__levels_5
Completed: baseline_6_variables__shock_2019__levels_3
Completed: baseline_6_variables__shock_2019__levels_4
Completed: baseline_6_variables__shock_2019__levels_5
Completed: baseline_plus_gdp_stability_2007__shock_2007__levels_5
Completed: baseline_plus_gdp_stability_2019__shock_2019__levels_5
Completed: sensitivity_remove_governance__shock_2007__levels_5
Completed: sensitivity_remove_governance__shock_2019__levels_5
Completed: sensitivity_replace_debt_with_productivity__shock_2007__levels_5
Completed: sensitivity_replace_debt_with_productivity__shock_2019__levels_5
Completed runs: 12


,run_key,set_name,shock_id,profile_levels,sample_strategy,require_recovery_available,country_count,profile_count,variable_count,variables,pareto_profile_count,pareto_country_count,dominance_relation_count,hasse_edge_count,layer_count,max_country_count_per_profile,mean_country_count_per_profile,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_run,hasse_diagram_path
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",1,1,110,54,7,1,1.0000,110,190,0.3667,0.6333,False,D:\Milano Bicocca\Course Materials\1st Year\02...
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,83,51,5,1,1.0000,83,217,0.2767,0.7233,False,D:\Milano Bicocca\Course Materials\1st Year\02...
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,84,54,5,1,1.0000,84,216,0.2800,0.7200,True,D:\Milano Bicocca\Course Materials\1st Year\02...
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,2019,3,complete_case,True,32,29,6,"debt_capacity_score_0_100,employment_strength_...",8,10,125,61,5,2,1.1034,125,281,0.3079,0.6921,False,D:\Milano Bicocca\Course Materials\1st Year\02...
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,13,126,62,5,2,1.0323,126,339,0.2710,0.7290,False,D:\Milano Bicocca\Course Materials\1st Year\02...
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,14,90,53,5,2,1.0323,90,375,0.1935,0.8065,True,D:\Milano Bicocca\Course Materials\1st Year\02...
6,baseline_plus_gdp_stability_2007__shock_2007__...,baseline_plus_gdp_stability_2007,2007,5,complete_case,True,25,25,7,"debt_capacity_score_0_100,employment_strength_...",11,11,57,41,4,1,1.0000,57,243,0.1900,0.8100,False,D:\Milano Bicocca\Course Materials\1st Year\02...
7,baseline_plus_gdp_stability_2019__shock_2019__...,baseline_plus_gdp_stability_2019,2019,5,complete_case,True,32,31,7,"debt_capacity_score_0_100,employment_strength_...",15,16,80,58,4,2,1.0323,80,385,0.1720,0.8280,False,D:\Milano Bicocca\Course Materials\1st Year\02...
8,sensitivity_remove_governance__shock_2007__lev...,sensitivity_remove_governance,2007,5,complete_case,True,25,25,5,"debt_capacity_score_0_100,employment_strength_...",8,8,96,52,5,1,1.0000,96,204,0.3200,0.6800,False,D:\Milano Bicocca\Course Materials\1st Year\02...
9,sensitivity_remove_governance__shock_2019__lev...,sensitivity_remove_governance,2019,5,complete_case,True,32,31,5,"debt_capacity_score_0_100,employment_strength_...",12,13,112,59,5,2,1.0323,112,353,0.2409,0.7591,False,D:\Milano Bicocca\Course Materials\1st Year\02...


In [8]:
# ------------------------------------------------------
# EXPORT COMBINED PROFILE POSET OUTPUTS
# ------------------------------------------------------

output_file_map = {
    "country_profile_map": "profile_country_map.csv",
    "profile_table": "profile_table.csv",
    "dominance_relations": "profile_dominance_relations.csv",
    "hasse_edges": "profile_hasse_edges.csv",
    "pareto_profiles": "profile_pareto_profiles.csv",
    "pareto_countries": "profile_pareto_countries.csv",
    "layer_summary": "profile_layer_summary.csv",
    "incomparability_pairs": "profile_incomparability_pairs.csv",
}

combined_output_tables = {}

for table_name, frames in combined_tables.items():
    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        # Keep a readable empty table instead of a blank 2-byte CSV.
        combined = pd.DataFrame({"message": ["No rows produced for this output. Check profile_run_errors.csv if this is unexpected."]})

    combined_output_tables[table_name] = combined

    output_name = output_file_map[table_name]

    combined.to_csv(
        PROCESSED_DIR / output_name,
        index=False,
    )

    combined.to_csv(
        DIAGNOSTICS_DIR / output_name,
        index=False,
    )

    print(f"{output_name}: {len(combined)} rows")

# Working main outputs as separate files for easier next-step use.
working_main_run_keys = (
    profile_run_summary.loc[
        profile_run_summary["is_working_main_run"].fillna(False).astype(bool),
        "run_key"
    ].tolist()
    if "is_working_main_run" in profile_run_summary.columns
    else []
) if not profile_run_summary.empty else []

for table_name, combined in combined_output_tables.items():
    if combined.empty or "run_key" not in combined.columns:
        continue

    main_subset = combined[combined["run_key"].isin(working_main_run_keys)].copy()
    main_subset.to_csv(
        PROCESSED_DIR / f"working_main_{output_file_map[table_name]}",
        index=False,
    )

print("Working main run keys:")
print(working_main_run_keys)

profile_country_map.csv: 349 rows
profile_table.csv: 341 rows
profile_dominance_relations.csv: 1231 rows
profile_hasse_edges.csv: 712 rows
profile_pareto_profiles.csv: 115 rows
profile_pareto_countries.csv: 120 rows
profile_layer_summary.csv: 61 rows
profile_incomparability_pairs.csv: 3502 rows
Working main run keys:
['baseline_6_variables__shock_2007__levels_5', 'baseline_6_variables__shock_2019__levels_5']


In [9]:
# ------------------------------------------------------
# WORKING MAIN RUN REVIEW TABLES
# ------------------------------------------------------

working_main_summary = profile_run_summary[
    profile_run_summary["is_working_main_run"]
].copy() if not profile_run_summary.empty else pd.DataFrame()

working_main_profile_table = combined_output_tables["profile_table"]
working_main_country_map = combined_output_tables["country_profile_map"]
working_main_pareto_countries = combined_output_tables["pareto_countries"]
working_main_layers = combined_output_tables["layer_summary"]

if not working_main_profile_table.empty:
    working_main_profile_table = working_main_profile_table[
        working_main_profile_table["run_key"].isin(working_main_run_keys)
    ].copy()

if not working_main_country_map.empty:
    working_main_country_map = working_main_country_map[
        working_main_country_map["run_key"].isin(working_main_run_keys)
    ].copy()

if not working_main_pareto_countries.empty:
    working_main_pareto_countries = working_main_pareto_countries[
        working_main_pareto_countries["run_key"].isin(working_main_run_keys)
    ].copy()

if not working_main_layers.empty:
    working_main_layers = working_main_layers[
        working_main_layers["run_key"].isin(working_main_run_keys)
    ].copy()

working_main_summary.to_csv(
    PROCESSED_DIR / "working_main_profile_run_summary.csv",
    index=False,
)

working_main_profile_table.to_csv(
    PROCESSED_DIR / "working_main_profile_table_review.csv",
    index=False,
)

working_main_country_map.to_csv(
    PROCESSED_DIR / "working_main_country_profile_map_review.csv",
    index=False,
)

working_main_pareto_countries.to_csv(
    PROCESSED_DIR / "working_main_pareto_countries_review.csv",
    index=False,
)

working_main_layers.to_csv(
    PROCESSED_DIR / "working_main_layer_summary_review.csv",
    index=False,
)

display(working_main_summary)
display(working_main_layers)
if not working_main_pareto_countries.empty:
    display(working_main_pareto_countries[["run_key", "shock_id", "Country", "country_name", "profile_id", "profile_code", "poset_layer", "Recovery"]].head(100))
else:
    display(pd.DataFrame({"message": ["No working main Pareto countries table available."]}))

,run_key,set_name,shock_id,profile_levels,sample_strategy,require_recovery_available,country_count,profile_count,variable_count,variables,pareto_profile_count,pareto_country_count,dominance_relation_count,hasse_edge_count,layer_count,max_country_count_per_profile,mean_country_count_per_profile,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_run,hasse_diagram_path
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,84,54,5,1,1.0000,84,216,0.2800,0.7200,True,D:\Milano Bicocca\Course Materials\1st Year\02...
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,14,90,53,5,2,1.0323,90,375,0.1935,0.8065,True,D:\Milano Bicocca\Course Materials\1st Year\02...


,run_key,set_name,shock_id,profile_levels,poset_layer,profile_count,countries_in_layer,profiles
12,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,1,8,8,P001;P002;P004;P006;P007;P008;P009;P010
13,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,2,8,8,P003;P005;P011;P012;P013;P014;P017;P019
14,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,3,5,5,P015;P016;P020;P022;P023
15,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,4,3,3,P018;P021;P024
16,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,5,1,1,P025
27,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,1,13,14,P001;P003;P004;P005;P006;P007;P008;P009;P010;P...
28,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,2,8,8,P002;P011;P016;P018;P019;P020;P021;P022
29,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,3,6,6,P017;P023;P024;P025;P026;P030
30,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,4,3,3,P027;P028;P029
31,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,5,1,1,P031


,run_key,shock_id,Country,country_name,profile_id,profile_code,poset_layer,Recovery
9,baseline_6_variables__shock_2007__levels_5,2007,CAN,Canada,P004,3-3-4-5-5-4,1,1.0000
10,baseline_6_variables__shock_2007__levels_5,2007,DNK,Denmark,P001,4-5-5-4-5-5,1,7.0000
11,baseline_6_variables__shock_2007__levels_5,2007,EST,Estonia,P002,5-5-2-5-5-4,1,8.0000
12,baseline_6_variables__shock_2007__levels_5,2007,FIN,Finland,P009,3-2-5-5-3-3,1,8.0000
13,baseline_6_variables__shock_2007__levels_5,2007,GBR,United Kingdom,P006,1-4-3-5-5-5,1,5.0000
14,baseline_6_variables__shock_2007__levels_5,2007,LUX,Luxembourg,P007,5-5-3-3-1-5,1,2.0000
15,baseline_6_variables__shock_2007__levels_5,2007,SVN,Slovenia,P008,5-4-3-2-4-3,1,8.0000
16,baseline_6_variables__shock_2007__levels_5,2007,USA,United States,P010,2-4-5-5-4-1,1,1.0000
40,baseline_6_variables__shock_2019__levels_5,2019,AUS,Australia,P004,3-3-3-5-5-5,1,0.0000
41,baseline_6_variables__shock_2019__levels_5,2019,CAN,Canada,P004,3-3-3-5-5-5,1,1.0000


In [10]:
# ------------------------------------------------------
# POSET QUALITY DIAGNOSTICS
# ------------------------------------------------------

quality_rows = []

for _, row in profile_run_summary.iterrows():
    warning_notes = []

    if row["country_count"] < 20:
        warning_notes.append("small_country_sample")

    if row["profile_count"] == row["country_count"]:
        warning_notes.append("each_country_unique_profile")

    if row["comparability_ratio"] < 0.20:
        warning_notes.append("very_low_comparability")

    if row["comparability_ratio"] > 0.90:
        warning_notes.append("very_high_comparability_possible_overcollapse")

    if row["pareto_country_count"] > row["country_count"] * 0.40:
        warning_notes.append("large_pareto_frontier")

    if row["max_country_count_per_profile"] > row["country_count"] * 0.35:
        warning_notes.append("large_profile_cluster")

    quality_rows.append({
        "run_key": row["run_key"],
        "set_name": row["set_name"],
        "shock_id": row["shock_id"],
        "profile_levels": row["profile_levels"],
        "country_count": row["country_count"],
        "profile_count": row["profile_count"],
        "pareto_country_count": row["pareto_country_count"],
        "comparability_ratio": row["comparability_ratio"],
        "incomparability_ratio": row["incomparability_ratio"],
        "layer_count": row["layer_count"],
        "quality_warnings": ";".join(warning_notes),
        "review_note": "Diagnostic only. Do not convert into scalar ranking.",
    })

profile_poset_quality_diagnostics = pd.DataFrame(quality_rows)

if profile_poset_quality_diagnostics.empty:
    profile_poset_quality_diagnostics = pd.DataFrame(columns=[
        "run_key", "set_name", "shock_id", "profile_levels", "country_count",
        "profile_count", "pareto_country_count", "comparability_ratio",
        "incomparability_ratio", "layer_count", "quality_warnings", "review_note"
    ])

profile_poset_quality_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "profile_poset_quality_diagnostics.csv",
    index=False,
)

display(profile_poset_quality_diagnostics)

,run_key,set_name,shock_id,profile_levels,country_count,profile_count,pareto_country_count,comparability_ratio,incomparability_ratio,layer_count,quality_warnings,review_note
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,25,25,1,0.3667,0.6333,7,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,25,25,8,0.2767,0.7233,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,25,25,8,0.2800,0.7200,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,2019,3,32,29,10,0.3079,0.6921,5,,Diagnostic only. Do not convert into scalar ra...
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,32,31,13,0.2710,0.7290,5,large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,32,31,14,0.1935,0.8065,5,very_low_comparability;large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
6,baseline_plus_gdp_stability_2007__shock_2007__...,baseline_plus_gdp_stability_2007,2007,5,25,25,11,0.1900,0.8100,4,each_country_unique_profile;very_low_comparabi...,Diagnostic only. Do not convert into scalar ra...
7,baseline_plus_gdp_stability_2019__shock_2019__...,baseline_plus_gdp_stability_2019,2019,5,32,31,16,0.1720,0.8280,4,very_low_comparability;large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
8,sensitivity_remove_governance__shock_2007__lev...,sensitivity_remove_governance,2007,5,25,25,8,0.3200,0.6800,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
9,sensitivity_remove_governance__shock_2019__lev...,sensitivity_remove_governance,2019,5,32,31,13,0.2409,0.7591,5,large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...


In [11]:
# ------------------------------------------------------
# DATA DICTIONARY AND METHODOLOGICAL NOTES
# ------------------------------------------------------

profile_poset_data_dictionary = pd.DataFrame([
    {
        "table": "profile_country_map.csv",
        "column": "profile_code",
        "description": "Ordinal profile code formed by the discretized candidate variable levels.",
    },
    {
        "table": "profile_country_map.csv",
        "column": "profile_id",
        "description": "Stable profile identifier within each run.",
    },
    {
        "table": "profile_table.csv",
        "column": "poset_layer",
        "description": "POSet layer. Layer 1 contains nondominated/Pareto profiles. This is not a scalar rank.",
    },
    {
        "table": "profile_table.csv",
        "column": "is_pareto_profile",
        "description": "True if no other profile dominates this profile.",
    },
    {
        "table": "profile_hasse_edges.csv",
        "column": "source_profile_id",
        "description": "Dominating profile in the transitive-reduced Hasse diagram edge.",
    },
    {
        "table": "profile_hasse_edges.csv",
        "column": "target_profile_id",
        "description": "Dominated profile in the transitive-reduced Hasse diagram edge.",
    },
    {
        "table": "profile_incomparability_pairs.csv",
        "column": "profile_a_id/profile_b_id",
        "description": "Profiles that are not comparable because neither dominates the other.",
    },
    {
        "table": "profile_run_summary.csv",
        "column": "comparability_ratio",
        "description": "Share of profile pairs that are comparable through the dominance relation.",
    },
])

profile_poset_data_dictionary.to_csv(
    PROCESSED_DIR / "profile_poset_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "No scalar index",
        "note": "The POSet structure is a partial order. Layers and profile IDs are not a final scalar ranking.",
    },
    {
        "topic": "Direction alignment",
        "note": "All variables used in the POSet are expected to be higher = better from the Pre-POSet EDA notebook.",
    },
    {
        "topic": "Discretization",
        "note": "Rank-quantile discretization is used to convert continuous scores into ordinal profile levels. Higher values receive higher levels.",
    },
    {
        "topic": "Dominance rule",
        "note": "Profile A dominates profile B if A is at least as high in every dimension and strictly higher in at least one dimension.",
    },
    {
        "topic": "Recovery",
        "note": "Recovery is carried only as a validation/outcome field and is not used in the dominance relation.",
    },
    {
        "topic": "Working main run",
        "note": "The baseline 6-variable 5-level run is a working default for code execution and review, not an irreversible methodological decision.",
    },
    {
        "topic": "Epsilon sensitivity",
        "note": "Epsilon sensitivity is not run here. It belongs in 08_Epsilon_Sensitivity_Country_Level.ipynb and 09_Epsilon_Margin_POSet_Robustness.ipynb.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "profile_poset_methodological_notes.csv",
    index=False,
)

display(profile_poset_data_dictionary)
display(methodological_notes)

,table,column,description
0,profile_country_map.csv,profile_code,Ordinal profile code formed by the discretized...
1,profile_country_map.csv,profile_id,Stable profile identifier within each run.
2,profile_table.csv,poset_layer,POSet layer. Layer 1 contains nondominated/Par...
3,profile_table.csv,is_pareto_profile,True if no other profile dominates this profile.
4,profile_hasse_edges.csv,source_profile_id,Dominating profile in the transitive-reduced H...
5,profile_hasse_edges.csv,target_profile_id,Dominated profile in the transitive-reduced Ha...
6,profile_incomparability_pairs.csv,profile_a_id/profile_b_id,Profiles that are not comparable because neith...
7,profile_run_summary.csv,comparability_ratio,Share of profile pairs that are comparable thr...


,topic,note
0,No scalar index,The POSet structure is a partial order. Layers...
1,Direction alignment,All variables used in the POSet are expected t...
2,Discretization,Rank-quantile discretization is used to conver...
3,Dominance rule,Profile A dominates profile B if A is at least...
4,Recovery,Recovery is carried only as a validation/outco...
5,Working main run,The baseline 6-variable 5-level run is a worki...
6,Epsilon sensitivity,Epsilon sensitivity is not run here. It belong...


In [12]:
# ------------------------------------------------------
# CREATE PROFILE POSET AUDIT WORKBOOK
# ------------------------------------------------------

PROFILE_POSET_AUDIT_XLSX = AUDIT_DIR / "07_Profile_POSet_Main_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_run_plan",
        "description": "All configured profile POSet runs.",
        "path": PROCESSED_DIR / "profile_run_plan.csv",
    },
    {
        "sheet_name": "02_run_summary",
        "description": "Summary metrics for all completed POSet runs.",
        "path": PROCESSED_DIR / "profile_run_summary.csv",
    },
    {
        "sheet_name": "03_run_errors",
        "description": "Errors encountered while running profile POSet configurations.",
        "path": DIAGNOSTICS_DIR / "profile_run_errors.csv",
    },
    {
        "sheet_name": "04_quality",
        "description": "POSet quality diagnostics and review warnings.",
        "path": DIAGNOSTICS_DIR / "profile_poset_quality_diagnostics.csv",
    },
    {
        "sheet_name": "05_working_main_summary",
        "description": "Working main run summary for baseline 6-variable 5-level runs.",
        "path": PROCESSED_DIR / "working_main_profile_run_summary.csv",
    },
    {
        "sheet_name": "06_working_main_layers",
        "description": "Layer summary for the working main runs.",
        "path": PROCESSED_DIR / "working_main_layer_summary_review.csv",
    },
    {
        "sheet_name": "07_working_main_pareto",
        "description": "Pareto countries in the working main runs.",
        "path": PROCESSED_DIR / "working_main_pareto_countries_review.csv",
    },
    {
        "sheet_name": "08_profile_table",
        "description": "All profile-level tables across runs.",
        "path": PROCESSED_DIR / "profile_table.csv",
    },
    {
        "sheet_name": "09_hasse_edges",
        "description": "All Hasse edges across runs.",
        "path": PROCESSED_DIR / "profile_hasse_edges.csv",
    },
    {
        "sheet_name": "10_dictionary",
        "description": "Profile POSet output data dictionary.",
        "path": PROCESSED_DIR / "profile_poset_data_dictionary.csv",
    },
    {
        "sheet_name": "11_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "profile_poset_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(PROFILE_POSET_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "07 Profile POSet Main Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for pure profile-level POSet runs.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Profile POSet audit workbook created:")
print(PROFILE_POSET_AUDIT_XLSX.resolve())

Profile POSet audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\07_Profile_POSet_Main_Audit.xlsx


In [14]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("07 PROFILE POSET MAIN COMPLETE")
print("=" * 80)

print("Processed folder:")
print(PROCESSED_DIR.resolve())

print("Diagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("Figures folder:")
print(FIGURES_DIR.resolve())

print("Audit workbook:")
print(PROFILE_POSET_AUDIT_XLSX.resolve())

print("Main processed outputs:")
main_outputs = [
    "profile_run_plan.csv",
    "profile_run_summary.csv",
    "profile_country_map.csv",
    "profile_table.csv",
    "profile_dominance_relations.csv",
    "profile_hasse_edges.csv",
    "profile_pareto_profiles.csv",
    "profile_pareto_countries.csv",
    "profile_layer_summary.csv",
    "profile_incomparability_pairs.csv",
    "working_main_profile_run_summary.csv",
    "working_main_country_profile_map_review.csv",
    "working_main_pareto_countries_review.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("Run summary:")
display(profile_run_summary)

print("Quality diagnostics:")
display(profile_poset_quality_diagnostics)

print("Important notes:")
print("1. Recovery was not used as an ordering variable.")
print("2. The baseline 5-level profile run is a working main output, not a final irreversible decision.")
print("3. Profile layers are not scalar rankings.")
print("4. Epsilon checks come next in notebook 08/09.")

print("\nNext notebook:")
print("08_Epsilon_Sensitivity_Country_Level.ipynb")

07 PROFILE POSET MAIN COMPLETE
Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Profile_POSet_Main
Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\07_Profile_POSet_Main
Figures folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Profile_POSet_Main
Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\07_Profile_POSet_Main_Audit.xlsx
Main processed outputs:
- FOUND: profile_run_plan.csv
- FOUND: profile_run_summary.csv
- FOUND: profile_country_map.csv
- FOUND: profile_table.csv
- FOUND: profile_dominance_relations.csv
- FOUND: profile_hasse_edges.csv
- FOUND: profile_pareto_profiles.csv
- FOUND: profile_pareto_countries.csv

,run_key,set_name,shock_id,profile_levels,sample_strategy,require_recovery_available,country_count,profile_count,variable_count,variables,pareto_profile_count,pareto_country_count,dominance_relation_count,hasse_edge_count,layer_count,max_country_count_per_profile,mean_country_count_per_profile,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_run,hasse_diagram_path
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",1,1,110,54,7,1,1.0000,110,190,0.3667,0.6333,False,D:\Milano Bicocca\Course Materials\1st Year\02...
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,83,51,5,1,1.0000,83,217,0.2767,0.7233,False,D:\Milano Bicocca\Course Materials\1st Year\02...
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,84,54,5,1,1.0000,84,216,0.2800,0.7200,True,D:\Milano Bicocca\Course Materials\1st Year\02...
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,2019,3,complete_case,True,32,29,6,"debt_capacity_score_0_100,employment_strength_...",8,10,125,61,5,2,1.1034,125,281,0.3079,0.6921,False,D:\Milano Bicocca\Course Materials\1st Year\02...
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,13,126,62,5,2,1.0323,126,339,0.2710,0.7290,False,D:\Milano Bicocca\Course Materials\1st Year\02...
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,14,90,53,5,2,1.0323,90,375,0.1935,0.8065,True,D:\Milano Bicocca\Course Materials\1st Year\02...
6,baseline_plus_gdp_stability_2007__shock_2007__...,baseline_plus_gdp_stability_2007,2007,5,complete_case,True,25,25,7,"debt_capacity_score_0_100,employment_strength_...",11,11,57,41,4,1,1.0000,57,243,0.1900,0.8100,False,D:\Milano Bicocca\Course Materials\1st Year\02...
7,baseline_plus_gdp_stability_2019__shock_2019__...,baseline_plus_gdp_stability_2019,2019,5,complete_case,True,32,31,7,"debt_capacity_score_0_100,employment_strength_...",15,16,80,58,4,2,1.0323,80,385,0.1720,0.8280,False,D:\Milano Bicocca\Course Materials\1st Year\02...
8,sensitivity_remove_governance__shock_2007__lev...,sensitivity_remove_governance,2007,5,complete_case,True,25,25,5,"debt_capacity_score_0_100,employment_strength_...",8,8,96,52,5,1,1.0000,96,204,0.3200,0.6800,False,D:\Milano Bicocca\Course Materials\1st Year\02...
9,sensitivity_remove_governance__shock_2019__lev...,sensitivity_remove_governance,2019,5,complete_case,True,32,31,5,"debt_capacity_score_0_100,employment_strength_...",12,13,112,59,5,2,1.0323,112,353,0.2409,0.7591,False,D:\Milano Bicocca\Course Materials\1st Year\02...


Quality diagnostics:


,run_key,set_name,shock_id,profile_levels,country_count,profile_count,pareto_country_count,comparability_ratio,incomparability_ratio,layer_count,quality_warnings,review_note
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,25,25,1,0.3667,0.6333,7,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,25,25,8,0.2767,0.7233,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,25,25,8,0.2800,0.7200,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,2019,3,32,29,10,0.3079,0.6921,5,,Diagnostic only. Do not convert into scalar ra...
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,32,31,13,0.2710,0.7290,5,large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,32,31,14,0.1935,0.8065,5,very_low_comparability;large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
6,baseline_plus_gdp_stability_2007__shock_2007__...,baseline_plus_gdp_stability_2007,2007,5,25,25,11,0.1900,0.8100,4,each_country_unique_profile;very_low_comparabi...,Diagnostic only. Do not convert into scalar ra...
7,baseline_plus_gdp_stability_2019__shock_2019__...,baseline_plus_gdp_stability_2019,2019,5,32,31,16,0.1720,0.8280,4,very_low_comparability;large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...
8,sensitivity_remove_governance__shock_2007__lev...,sensitivity_remove_governance,2007,5,25,25,8,0.3200,0.6800,5,each_country_unique_profile,Diagnostic only. Do not convert into scalar ra...
9,sensitivity_remove_governance__shock_2019__lev...,sensitivity_remove_governance,2019,5,32,31,13,0.2409,0.7591,5,large_pareto_frontier,Diagnostic only. Do not convert into scalar ra...


Important notes:
1. Recovery was not used as an ordering variable.
2. The baseline 5-level profile run is a working main output, not a final irreversible decision.
3. Profile layers are not scalar rankings.
4. Epsilon checks come next in notebook 08/09.

Next notebook:
08_Epsilon_Sensitivity_Country_Level.ipynb
